In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPool2D,Conv2D,BatchNormalization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
IDG=ImageDataGenerator(rescale=1./255,rotation_range=20,zoom_range=0.2,horizontal_flip=True,vertical_flip=True,validation_split=0.2,fill_mode='nearest')


In [4]:
train=IDG.flow_from_directory("LIver Ultrasound Image (Fatty Liver)/.",target_size=(224,224),batch_size=16,class_mode='categorical',subset='training',shuffle=True)

Found 551 images belonging to 3 classes.


In [5]:
val=IDG.flow_from_directory("LIver Ultrasound Image (Fatty Liver)/.",target_size=(224,224),batch_size=16,class_mode='categorical',subset='validation',shuffle=False)

Found 136 images belonging to 3 classes.


In [6]:
print(f"train shape: {train.samples}")
print(f"val shape: {val.samples}")
print(f"classes shape: {train.class_indices}")

train shape: 551
val shape: 136
classes shape: {'Mild': 0, 'Normal': 1, 'Severe': 2}


In [7]:
print("train shape",train.image_shape)
print("val shape",val.image_shape)

train shape (224, 224, 3)
val shape (224, 224, 3)


In [8]:
model=Sequential()
model.add(Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPool2D(pool_size=(2,2)))
model.add(Flatten())
model.add(Dense(3,activation='softmax'))

C:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
compile=model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),loss='categorical_crossentropy',metrics=['accuracy'])


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,635,459 (21.50 MB)

 Trainable params: 5,635,459 (21.50 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
imgs,lbls=[],[]
for i in range(len(val)):
    x,y=val[i]
    imgs.append(x)
    lbls.append(y)

val_images=np.concatenate(imgs)
val_labels=np.concatenate(lbls)

In [12]:
val_img,test_img=val_images[:68],val_images[68:]
val_lbl,test_lbl=val_labels[:68],val_labels[68:]

In [13]:
history=model.fit(train,validation_data=(val_img,val_lbl),epochs=30,batch_size=16)

Epoch 1/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 184ms/step - accuracy: 0.3448 - loss: 1.0943 - val_accuracy: 0.0147 - val_loss: 1.2898
Epoch 2/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 185ms/step - accuracy: 0.3811 - loss: 1.0791 - val_accuracy: 0.0441 - val_loss: 1.6569
Epoch 3/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 12s 332ms/step - accuracy: 0.4392 - loss: 1.0690 - val_accuracy: 0.0441 - val_loss: 2.0653
Epoch 4/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 9s 251ms/step - accuracy: 0.4265 - loss: 1.0601 - val_accuracy: 0.0147 - val_loss: 1.8936
Epoch 5/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 7s 188ms/step - accuracy: 0.4555 - loss: 1.0570 - val_accuracy: 0.0441 - val_loss: 2.6469
Epoch 6/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 8s 218ms/step - accuracy: 0.4465 - loss: 1.0635 - val_accuracy: 0.0441 - val_loss: 2.9161
Epoch 7/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 12s 252ms/step - accuracy: 0.4973 - loss: 1.0313 - val_accuracy: 0.0441 - val_loss: 3.4789
Epoch 8/30
35/35 ━━━━━━━━━━━━━━━━━━━━ 10s 300ms/step - accuracy: 0.4773 - loss: 1.0245 - val_accuracy: